In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('smart_home_energy_dataset.csv')

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Check missing values
print(df.isnull().sum())

# Convert numeric columns
numeric_cols = [
    'appliance_usage_hours',
    'ac_usage_hours',
    'heater_usage_hours',
    'energy_consumption_kwh',
    'avg_temperature_c', 
    'humidity_pct',
    'solar_generation_kwh',
    'electricity_price',
    'num_occupants'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert categorical columns
categorical_cols = ['weekday', 'peak_hour_usage']

for col in categorical_cols:
    df[col] = df[col].astype('category')

# Convert datetime column
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Verify
print(df.dtypes)

# Numeric columns → fill with median
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns → fill with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Datetime → drop rows if missing (rare case)
df.dropna(subset=['date'], inplace=True)

df.dropna(inplace=True)

import numpy as np

def treat_outliers(col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap the outliers
    df[col] = np.where(df[col] < lower, lower,
                       np.where(df[col] > upper, upper, df[col]))

# Apply to required columns
outlier_cols = ['appliance_usage_hours', 'ac_usage_hours', 'energy_consumption_kwh']

for col in outlier_cols:
    treat_outliers(col)

    df['total_usage_hours'] = (
    df['appliance_usage_hours'] +
    df['ac_usage_hours'] +
    df['heater_usage_hours']
)

df['temperature_effect'] = df['avg_temperature_c'] * df['ac_usage_hours']

df['net_energy_usage'] = (
    df['energy_consumption_kwh'] -
    df['solar_generation_kwh']
)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

categorical_cols = ['weekday', 'peak_hour_usage']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(['energy_consumption_kwh', 'date', 'household_id'], axis=1)
y = df['energy_consumption_kwh']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# Linear Regression
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Decision Tree
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Random Forest
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    return mae, rmse, r2

lr_metrics = evaluate(y_test, y_pred_lr)
dt_metrics = evaluate(y_test, y_pred_dt)
rf_metrics = evaluate(y_test, y_pred_rf)

print("Linear Regression:", lr_metrics)
print("Decision Tree:", dt_metrics)
print("Random Forest:", rf_metrics)

import pandas as pd

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'MAE': [lr_metrics[0], dt_metrics[0], rf_metrics[0]],
    'RMSE': [lr_metrics[1], dt_metrics[1], rf_metrics[1]],
    'R2 Score': [lr_metrics[2], dt_metrics[2], rf_metrics[2]]
})

print(results)

date                      22
household_id              16
num_occupants             16
avg_temperature_c         19
appliance_usage_hours     14
ac_usage_hours            18
heater_usage_hours        15
solar_generation_kwh      23
humidity_pct              17
weekday                   15
electricity_price         24
peak_hour_usage           10
energy_consumption_kwh    21
dtype: int64
date                      datetime64[ns]
household_id                      object
num_occupants                    float64
avg_temperature_c                float64
appliance_usage_hours            float64
ac_usage_hours                   float64
heater_usage_hours               float64
solar_generation_kwh             float64
humidity_pct                     float64
weekday                         category
electricity_price                float64
peak_hour_usage                 category
energy_consumption_kwh           float64
dtype: object


C:\Users\karan\AppData\Local\Temp\ipykernel_22288\2424677781.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\karan\AppData\Local\Temp\ipykernel_22288\2424677781.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

Linear Regression: (3.4065058466344805e-14, np.float64(4.390554540104155e-14), 1.0)
Decision Tree: (2.6482875055057167, np.float64(3.47863851366751), 0.8976647934382853)
Random Forest: (1.5512583103366275, np.float64(2.020386485183765), 0.9654796350712604)
               Model           MAE          RMSE  R2 Score
0  Linear Regression  3.406506e-14  4.390555e-14  1.000000
1      Decision Tree  2.648288e+00  3.478639e+00  0.897665
2      Random Forest  1.551258e+00  2.020386e+00  0.965480


# Question – Statistical Analysis (Regression Insight)
- Scenario:
    The company wants to statistically validate whether environmental factors significantly affect energy consumption.
- Required Tasks:
    - Perform a hypothesis test: H0: Temperature has no effect on energy consumption
    - Apply: Choose an appropriate statistical test and justify
    - Compute: 95% confidence interval for average energy consumption
    - Perform: Correlation analysis between:
        - temperature
        - appliance usage
        - energy consumption
    - Explain: Overfitting vs Underfitting in regression models

In [3]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(df['avg_temperature_c'], df['energy_consumption_kwh'])

print("Correlation:", corr)
print("P-value:", p_value)

Correlation: 0.11989652474888199
P-value: 0.030960098620100945


In [4]:
import numpy as np
from scipy import stats

mean = df['energy_consumption_kwh'].mean()
sem = stats.sem(df['energy_consumption_kwh'])  # Standard error

ci = stats.t.interval(0.95, len(df)-1, loc=mean, scale=sem)

print("Mean:", mean)
print("95% Confidence Interval:", ci)

Mean: 26.8497230511538
95% Confidence Interval: (np.float64(25.547446291550692), np.float64(28.151999810756905))


In [5]:
corr_matrix = df[['avg_temperature_c', 'appliance_usage_hours', 'energy_consumption_kwh']].corr()

print(corr_matrix)

                        avg_temperature_c  appliance_usage_hours  \
avg_temperature_c                1.000000              -0.107915   
appliance_usage_hours           -0.107915               1.000000   
energy_consumption_kwh           0.119897              -0.037736   

                        energy_consumption_kwh  
avg_temperature_c                     0.119897  
appliance_usage_hours                -0.037736  
energy_consumption_kwh                1.000000  


# Overfitting
- Overfitting occurs when a model learns noise and specific patterns in the training data, resulting in excellent training performance but poor generalization to new data.
- Model learns too much (including noise)
- Very high training accuracy
- Poor test performance

# Underfitting
- Underfitting occurs when a model is too simple to capture underlying patterns, resulting in poor performance on both training and testing data.
- Model is too simple
- Poor training performance
- Poor test performance